<a href="https://colab.research.google.com/github/anatormin/Estrutura-De-Dados-II/blob/main/grafosaero.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [3]:
# -*- coding: utf-8 -*-
"""grafosaero.ipynb

Trabalho de Estrutura de Dados II — Ana Luísa Tormin Vaz — RGM: 47881534
"""

from collections import defaultdict, deque


# ==============================================================================
# 1. MODELO
# ==============================================================================
# Grafo não direcionado e sem pesos.
# Cada vértice é um aeroporto (código IATA) e cada aresta é uma rota direta.
#
# CGH Congonhas | GRU Guarulhos | VCP Viracopos | BSB Brasília
# CNF Confins   | SSA Salvador  | REC Recife    | FOR Fortaleza
# ==============================================================================

ROTAS = [
    ("CGH", "GRU"), ("CGH", "VCP"), ("GRU", "VCP"), ("GRU", "BSB"), ("VCP", "BSB"),
    ("BSB", "CNF"), ("BSB", "SSA"), ("CNF", "SSA"),
    ("SSA", "REC"), ("SSA", "FOR"), ("REC", "FOR"),
]


class RedeVoos:
    """
    Representa uma rede de voos como grafo não direcionado.

    A estrutura usada é uma lista de adjacência (dict de listas), escolhida
    porque o grafo é esparso: ocupa O(V+E) de memória e percorrer os
    vizinhos de um vértice é direto.

    As rotas bloqueadas são armazenadas em um conjunto de frozensets,
    o que permite bloquear e desbloquear sem reconstruir o grafo.
    """

    def __init__(self, rotas):
        # defaultdict(list) cria automaticamente uma lista vazia para
        # cada novo aeroporto — evita o uso de setdefault manualmente.
        self.adjacencia = defaultdict(list)
        self.rotas_bloqueadas = set()

        for a, b in rotas:
            self.adjacencia[a].append(b)
            self.adjacencia[b].append(a)

    # --------------------------------------------------------------------------
    # Operações de bloqueio
    # --------------------------------------------------------------------------

    def bloquear_rota(self, a, b):
        """Marca a rota entre a e b como indisponível."""
        self.rotas_bloqueadas.add(frozenset((a, b)))

    def desbloquear_rota(self, a, b):
        """Remove o bloqueio da rota entre a e b, se existir."""
        self.rotas_bloqueadas.discard(frozenset((a, b)))

    def vizinhos_disponiveis(self, v):
        """Retorna apenas os vizinhos cuja rota não está bloqueada."""
        return [
            w for w in self.adjacencia[v]
            if frozenset((v, w)) not in self.rotas_bloqueadas
        ]

    # --------------------------------------------------------------------------
    # Reconstrução de caminho (usada por BFS e DFS)
    # --------------------------------------------------------------------------

    @staticmethod
    def reconstruir_caminho(pais, destino):
        """
        Reconstrói o caminho do destino até a origem usando o dicionário
        de pais, e então inverte para retornar na ordem correta.
        """
        caminho = []
        atual = destino
        while atual is not None:
            caminho.append(atual)
            atual = pais[atual]
        return caminho[::-1]

    # --------------------------------------------------------------------------
    # BFS — Busca em Largura
    # --------------------------------------------------------------------------
    # Usa fila (FIFO): explora por camadas. Em grafo sem pesos, o primeiro
    # caminho que chega ao destino é o de MENOR número de rotas.
    # --------------------------------------------------------------------------

    def caminho_bfs(self, origem, destino):
        """
        Retorna uma tupla (caminho, ordem_de_visita).
        Se não houver caminho, caminho é None.
        """
        pais = {origem: None}
        fila = deque([origem])
        ordem_visita = []

        while fila:
            atual = fila.popleft()
            ordem_visita.append(atual)

            if atual == destino:
                return self.reconstruir_caminho(pais, destino), ordem_visita

            for vizinho in self.vizinhos_disponiveis(atual):
                if vizinho not in pais:
                    pais[vizinho] = atual
                    fila.append(vizinho)

        return None, ordem_visita

    # --------------------------------------------------------------------------
    # DFS — Busca em Profundidade
    # --------------------------------------------------------------------------
    # Usa pilha (LIFO): vai fundo por um ramo antes de voltar. Acha UM caminho,
    # não necessariamente o mais curto. Versão iterativa para evitar o limite
    # de recursão do Python em grafos grandes.
    # --------------------------------------------------------------------------

    def caminho_dfs(self, origem, destino):
        """
        Retorna uma tupla (caminho, ordem_de_visita).
        Se não houver caminho, caminho é None.
        """
        pais = {origem: None}
        pilha = [origem]
        visitados = set()
        ordem_visita = []

        while pilha:
            atual = pilha.pop()

            if atual in visitados:
                continue

            visitados.add(atual)
            ordem_visita.append(atual)

            if atual == destino:
                return self.reconstruir_caminho(pais, destino), ordem_visita

            # reversed() para empilhar os vizinhos de modo que o primeiro
            # da lista seja visitado primeiro (mantém a ordem de cadastro).
            for vizinho in reversed(self.vizinhos_disponiveis(atual)):
                if vizinho not in visitados:
                    pais.setdefault(vizinho, atual)
                    pilha.append(vizinho)

        return None, ordem_visita

    # --------------------------------------------------------------------------
    # Utilitário: verifica se existe caminho entre dois vértices
    # --------------------------------------------------------------------------

    def existe_caminho(self, origem, destino):
        """Retorna True se houver caminho disponível entre origem e destino."""
        caminho, _ = self.caminho_bfs(origem, destino)
        return caminho is not None


# ==============================================================================
# 2. SIMULAÇÃO
# ==============================================================================

def formatar_caminho(caminho):
    """Formata uma lista de vértices como 'A -> B -> C'."""
    return " -> ".join(caminho)


def executar_cenario(titulo, rede, origem, destino):
    """
    Executa BFS e DFS para um cenário e imprime os resultados de forma
    organizada: caminho encontrado, número de rotas e ordem de visita.
    """
    print(f"\n{'=' * 60}")
    print(f"  {titulo}")
    print(f"  Origem: {origem} | Destino: {destino}")
    print(f"{'=' * 60}")

    for nome, funcao in (("BFS", rede.caminho_bfs), ("DFS", rede.caminho_dfs)):
        caminho, ordem = funcao(origem, destino)

        if caminho:
            num_rotas = len(caminho) - 1
            print(f"  {nome}: {formatar_caminho(caminho)}  [{num_rotas} rotas]")
        else:
            print(f"  {nome}: nenhum caminho disponível")

        print(f"       ordem de visita: {', '.join(ordem)}")

    # Informação extra: verifica se o destino é alcançável
    if rede.existe_caminho(origem, destino):
        print(f"\n  ✔  Destino {destino} alcançável a partir de {origem}.")
    else:
        print(f"\n  ✘  Destino {destino} NÃO alcançável a partir de {origem}.")


# ==============================================================================
# 3. PROGRAMA PRINCIPAL
# ==============================================================================

if __name__ == "__main__":
    rede = RedeVoos(ROTAS)

    print(f"Rede de voos: {len(rede.adjacencia)} aeroportos | "
          f"{len(ROTAS)} rotas cadastradas")

    origem = "CGH"
    destino = "FOR"

    # --------------------------------------------------------------------------
    # Cenário 1: Rede completa, sem bloqueios
    # --------------------------------------------------------------------------
    executar_cenario("CENÁRIO 1 — Rede completa (sem bloqueios)",
                     rede, origem, destino)

    # --------------------------------------------------------------------------
    # Cenário 2: Rota BSB-SSA bloqueada (desvio possível por CNF)
    # --------------------------------------------------------------------------
    rede.bloquear_rota("BSB", "SSA")
    print("\n>>> Rota bloqueada: BSB <-> SSA")

    executar_cenario("CENÁRIO 2 — Rota BSB-SSA bloqueada",
                     rede, origem, destino)

    # --------------------------------------------------------------------------
    # Cenário 3: BSB-SSA e BSB-CNF bloqueadas (grafo desconectado)
    # --------------------------------------------------------------------------
    rede.bloquear_rota("BSB", "CNF")
    print("\n>>> Rotas bloqueadas: BSB <-> SSA e BSB <-> CNF")

    executar_cenario("CENÁRIO 3 — Grafo desconectado",
                     rede, origem, destino)

    # --------------------------------------------------------------------------
    # Extra: desbloquear tudo e confirmar que a rede volta ao normal
    # --------------------------------------------------------------------------
    rede.desbloquear_rota("BSB", "SSA")
    rede.desbloquear_rota("BSB", "CNF")
    print("\n>>> Todas as rotas desbloqueadas.")

    executar_cenario("CENÁRIO 4 — Rede restaurada",
                     rede, origem, destino)

Aeroportos: 8 | Rotas: 11

--- ANTES do bloqueio (CGH -> FOR) ---
BFS: CGH -> GRU -> BSB -> SSA -> FOR  [4 rotas]
     ordem de visita: CGH, GRU, VCP, BSB, CNF, SSA, REC, FOR
DFS: CGH -> GRU -> BSB -> SSA -> FOR  [4 rotas]
     ordem de visita: CGH, GRU, VCP, BSB, CNF, SSA, REC, FOR

>>> Rota indisponível: BSB <-> SSA

--- DEPOIS do bloqueio (CGH -> FOR) ---
BFS: CGH -> GRU -> BSB -> CNF -> SSA -> FOR  [5 rotas]
     ordem de visita: CGH, GRU, VCP, BSB, CNF, SSA, REC, FOR
DFS: CGH -> GRU -> BSB -> CNF -> SSA -> FOR  [5 rotas]
     ordem de visita: CGH, GRU, VCP, BSB, CNF, SSA, REC, FOR

>>> Rotas indisponíveis: BSB <-> SSA e BSB <-> CNF

--- Cenário extremo (grafo desconectado) (CGH -> FOR) ---
BFS: nenhum caminho disponível
     ordem de visita: CGH, GRU, VCP, BSB
DFS: nenhum caminho disponível
     ordem de visita: CGH, GRU, VCP, BSB
